# 03 評価の不確かさ — 「差がある」と言えるのは、いつか

## このノートの目的
- モデルを比べるとき、**2種類のばらつき**があることを知る。
  1. **乱数（シード）によるばらつき**: 同じデータでも、乱数を変えると結果が少し変わる。
  2. **データの偶然によるばらつき**: 評価に使ったデータが、たまたまそうだった。
- 後者を見積もる方法、**ブートストラップ**を使えるようになる。
- 陽性が少ないと、評価の**信頼区間**が非常に広くなることを体験する。

## キーワード
| 用語 | やさしい説明 |
|---|---|
| **シード（乱数の種）** | 乱数を固定するための数字。変えると、木の作られ方が変わる。 |
| **ブートストラップ** | 手元のデータから**復元抽出**で何度も作り直し、そのたびに指標を計算して、**指標のばらつき**を見積もる方法。 |
| **信頼区間（95%区間）** | 「本当の値は、おおよそこの範囲にある」という幅。ブートストラップなら、結果の**中央95%**の範囲。 |
| **日単位で再抽出** | 同じ日の3標本は同じ答えなので、**日ごと**まとめて入れ替える（標本ごとだと、独立だと思い込んで区間が狭くなりすぎる）。 |

In [ ]:
# 準備: 必要なライブラリを読み込み、src フォルダの部品（説明変数を作る dataset.py など）を使えるようにします
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve() / "src"))   # ノートブックは notebooks/ にあるので、1つ上の src を探す
plt.rcParams["font.family"] = ["Hiragino Sans", "Arial Unicode MS", "sans-serif"]  # 日本語を表示するための設定
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 140)
import dataset as ds   # 説明変数と正解を作る部品（src/dataset.py）

In [ ]:
# 目的: 評価に使う予測（leave-one-year-out の保持予測）を用意する。ExtraTrees と ランダムフォレストの両方（木は100本に減らして軽くしてある）
from sklearn.metrics import average_precision_score
import run_phase2 as p2

x = ds.build("naha"); f = ds.features("naha", change=False); target = "y20_region"
cfg = {**p2.BASE, "min_samples_leaf": 20, "n_estimators": 100, "change": False}
use = (x.year.isin(range(2010, 2025)) & x.wind_ok).to_numpy()
xe = x[use].reset_index(drop=True)
y, days, years = xe[target].to_numpy(), xe.date.to_numpy(), xe.year.to_numpy()
print(f"標本 {len(y)}、陽性 {y.sum()}（{xe.loc[y == 1, 'date'].nunique()} 日）")

def oof_use(kind, seed):
    return p2.oof(x, f, target, kind, cfg, seed)[use]          # その年を学習に使わずに予測 → 評価対象だけ取り出す

et = {s: oof_use("et", s) for s in range(8)}                    # シードを8通り
rf = {s: oof_use("rf", s) for s in range(8)}
print("予測を作りました")

## 1. シードによるばらつき

同じデータ・同じ設定で、**乱数のシードだけ**変えて8回学習し直したときの PR-AUC を見ます。

In [ ]:
# 目的: シードを変えたときの PR-AUC のばらつき（標準偏差）を見る
ap_et = np.array([average_precision_score(y, et[s]) for s in et])
ap_rf = np.array([average_precision_score(y, rf[s]) for s in rf])
print(f"ExtraTrees  : 平均 {ap_et.mean():.3f}、標準偏差 {ap_et.std():.4f}（範囲 {ap_et.min():.3f}〜{ap_et.max():.3f}）")
print(f"RandomForest: 平均 {ap_rf.mean():.3f}、標準偏差 {ap_rf.std():.4f}（範囲 {ap_rf.min():.3f}〜{ap_rf.max():.3f}）")

**出力の見方**: シードを変えたときのばらつき（標準偏差）は**0.00x**程度で、とても小さいです。
「ExtraTrees は 0.088、RandomForest は 0.090」のような差は、**シードのばらつきと同じくらい**です。では、**データの偶然**も含めると、どうでしょう。

## 2. データの偶然 — ブートストラップ

評価データは「たまたまこの年の、この日々」でした。**別の年だったら**、PR-AUC はどれくらい変わったでしょうか。
手元の日々を**復元抽出**で何度も選び直し、そのたびに PR-AUC を計算して、**ばらつき**を見ます。

In [ ]:
# 目的: 日単位のブートストラップで、PR-AUC とその差の95%区間を求める
def bootstrap(y, days, preds, B=1000, seed=0):
    # preds: {名前: 予測} → {名前: B個のPR-AUC}。同じ日の標本はまとめて入れ替える
    rng = np.random.default_rng(seed)
    udays = np.unique(days)
    idx = {d: np.where(days == d)[0] for d in udays}
    out = {k: [] for k in preds}
    for _ in range(B):
        pick = np.concatenate([idx[d] for d in rng.choice(udays, len(udays))])   # 日を、同じ数だけ復元抽出
        if y[pick].sum() == 0:
            continue
        for k, p in preds.items():
            out[k].append(average_precision_score(y[pick], p[pick]))
    return {k: np.array(v) for k, v in out.items()}

P = {"ExtraTrees": np.mean(list(et.values()), axis=0), "RandomForest": np.mean(list(rf.values()), axis=0)}   # 8シードの平均予測
b = bootstrap(y, days, P)
for k, v in b.items():
    lo, hi = np.percentile(v, [2.5, 97.5])
    print(f"{k:13s}: PR-AUC {average_precision_score(y, P[k]):.3f}   95%区間 {lo:.3f}〜{hi:.3f}（幅 {hi - lo:.3f}）")
diff = b["ExtraTrees"] - b["RandomForest"]
lo, hi = np.percentile(diff, [2.5, 97.5])
print(f"差（ExtraTrees − RandomForest）: 平均 {diff.mean():+.3f}、95%区間 {lo:+.3f}〜{hi:+.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.3))
for k, v in b.items():
    ax[0].hist(v, bins=40, alpha=0.5, label=k)
ax[0].set_xlabel("PR-AUC（ブートストラップ）"); ax[0].legend()
ax[1].hist(diff, bins=40, color="gray"); ax[1].axvline(0, color="r"); ax[1].set_xlabel("差（ExtraTrees − RandomForest）")
plt.tight_layout(); plt.show()

**出力の見方**
- **95%区間の幅**（約0.1）は、シードによるばらつき（標準偏差 約0.005）の**約20倍**です。**データの偶然のほうが圧倒的に大きい**ことが分かります。
- 差の95%区間が **0 をまたいでいる**（例: −0.02〜+0.03）なら、「どちらが良いとは言えない」です。**平均の差がプラスでも、区間が0をまたぐなら、偶然かもしれません**。
- 右の図で、赤い線（0）が分布の真ん中あたりにあれば、「どちらが上になるかは、評価データ次第」という意味です。

## 3. 陽性が減ると、区間はどう広がるか

陽性の日を**ランダムに間引いて**、評価の信頼区間がどうなるかを見ます。

In [ ]:
# 目的: 陽性の日を 100%・50%・25% に間引いたデータで、ExtraTrees の PR-AUC の95%区間の幅を比べる
rng = np.random.default_rng(1)
pos_days = np.unique(days[y == 1]); neg_mask = ~np.isin(days, pos_days)
rows = []
for frac in (1.0, 0.5, 0.25):
    keep = rng.choice(pos_days, int(len(pos_days) * frac), replace=False)
    m = neg_mask | np.isin(days, keep)                                    # 陽性でない日 ＋ 残した陽性の日
    bb = bootstrap(y[m], days[m], {"ET": P["ExtraTrees"][m]}, B=500)["ET"]
    lo, hi = np.percentile(bb, [2.5, 97.5])
    rows.append({"残した陽性の日": len(keep), "PR-AUC": average_precision_score(y[m], P["ExtraTrees"][m]), "95%区間の下限": lo, "上限": hi, "幅": hi - lo,
                 "幅 ÷ PR-AUC": (hi - lo) / average_precision_score(y[m], P["ExtraTrees"][m])})
pd.DataFrame(rows).round(3)

**出力の見方**: 陽性の日を減らすと、**PR-AUC の値そのものが下がります**（陽性率が基準なので、陽性が減ると、全体が小さくなる）。区間の「幅」の絶対値も小さくなりますが、
**「幅 ÷ PR-AUC」（値に対する相対的な幅）は増えていきます**（約1.4倍 → 約1.7倍 → 約2.4倍）。つまり、**陽性が少ないほど、値の信頼性は相対的に落ちます**。
2025年の最終評価は、陽性が**5〜6日**しかありませんでした。区間が非常に広かった（那覇の ExtraTrees で 0.04〜0.40）のは、このためです。

## 4. 年ごとの PR-AUC — 1年だけで評価すると

実際に、**1年だけ**で評価すると、PR-AUC はどれくらいぶれるでしょうか。年ごとに計算します。

In [ ]:
# 目的: 年ごとの PR-AUC（その年の標本だけで計算）と、その年の陽性の日数を見る
rows = []
for yr in range(2010, 2025):
    m = years == yr
    npos = int(xe.loc[m & (y == 1), "date"].nunique())
    rows.append({"年": yr, "陽性の日": npos, "陽性率": y[m].mean(), "PR-AUC": average_precision_score(y[m], P["ExtraTrees"][m]) if npos else np.nan})
yy = pd.DataFrame(rows).set_index("年")
fig, ax = plt.subplots(figsize=(9, 3.3))
ax.bar(yy.index, yy["PR-AUC"], color="#4c72b0"); ax.plot(yy.index, yy["陽性率"], "r_", ms=18, mew=2, label="陽性率（基準）")
for i, r in yy.iterrows():
    ax.text(i, 0.002, f"{int(r['陽性の日'])}日", ha="center", fontsize=8, color="white")
ax.set_ylabel("PR-AUC"); ax.legend(); ax.set_xticks(yy.index); ax.tick_params(axis="x", rotation=60); plt.tight_layout(); plt.show()
print(f"年ごとの PR-AUC: 最小 {yy['PR-AUC'].min():.3f}、最大 {yy['PR-AUC'].max():.3f}、標準偏差 {yy['PR-AUC'].std():.3f}")

**出力の見方**: 棒の中の数字が**その年の陽性の日数**です。年によって PR-AUC は**約0.04〜0.60と大きくぶれ**、陽性の日が少ない年ほど不安定です。
**「2025年の1年だけ」で手法の優劣を決められない**のは、このためです。複数年をまとめた leave-one-year-out の評価と、日単位ブートストラップの区間を、**併せて見る**必要があります。

## まとめ
- ばらつきには、**シード（小さい）**と**データの偶然（大きい）**がある。比べるときは、**後者**（ブートストラップ）を見る。
- 差の95%区間が **0 をまたぐなら、「差がある」と言えない**。
- **陽性が少ないほど、区間は広がる**。評価は、陽性の件数を必ず併記する。
- 同じ日の標本は**日単位**で入れ替える（標本単位だと、区間が狭くなりすぎる）。

## 演習
1. **2.** の `B=1000` を `B=200` に変えてください（観察点: 区間の幅は変わるか。繰り返しが少ないと、区間の端が不安定になる）。
2. **2.** の `P` に、`"直近6時間降水量": x[use]["rain6"].fillna(0).to_numpy()` を追加して、ExtraTrees との差の区間を見てください（観察点: 単純な指標との差も、0をまたぐか）。
3. **3.** の `(1.0, 0.5, 0.25)` に `0.1` を加えてください（観察点: 陽性の日が数日になると、区間の幅はいくつになるか）。
4. **1.** で、`range(8)` のシード数を変えたとき、標準偏差がどう変わるか確認してください（`et` と `rf` の作成部分の `range(8)` を `range(3)` に）。